# Phase 9.1: Druggability Feature Engineering
## DNA Gene Mapping Project
**Author:** Sharique Mohammad  
**Date:** February 2026  

---

## Objective
Create gene-level features for druggability prediction:
- Aggregate variant-level data to gene-level
- Create gene-level features from pharmacogene data
- Prepare dataset for regression model

## Notebook Location
- Path: ml_phase/05_pharmacogene/05a_druggability_features.ipynb
- Figures: data/analytical/figures/phase3/ (continue numbering from 33)
- Data: data/ml/pharmacogene/

---
## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, str(Path.cwd().parent.parent))
from config import PROJECT_ROOT, DATA_DIR, MODEL_DIR, FIGURES_DIR, MODEL_CONFIG
from scripts.model_utils import save_model_metadata, print_summary

import psycopg2
from config import DATABASE_CONFIG

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("="*80)
print("PHASE 9.1: DRUGGABILITY FEATURE ENGINEERING")
print("="*80)
print(f"Random state: {MODEL_CONFIG['random_state']}")

---
## 2. Load Data from PostgreSQL

In [ ]:
print("\nConnecting to PostgreSQL...")

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG['database'],
    user=DATABASE_CONFIG['user'],
    password=DATABASE_CONFIG['password'],
    host=DATABASE_CONFIG['host'],
    port=DATABASE_CONFIG['port']
)

print("OK Connected to PostgreSQL")

In [ ]:
# Load pharmacogene features (has druggability scores)
print("\nLoading pharmacogene data...")

query = """
SELECT 
    gene_symbol,
    gene_id,
    is_pharmacogene,
    is_drug_target,
    is_kinase,
    is_receptor,
    is_transporter,
    druggability_score,
    enhanced_druggability_score,
    COUNT(*) as variant_count
FROM pharmacogene_ml_features
GROUP BY gene_symbol, gene_id, is_pharmacogene, is_drug_target, 
         is_kinase, is_receptor, is_transporter, druggability_score, 
         enhanced_druggability_score
LIMIT 1000
"""

pharmacogene_df = pd.read_sql(query, conn)
print(f"Loaded {len(pharmacogene_df):,} genes with pharmacogene data")
print(f"\nSample:")
print(pharmacogene_df.head())

---
## 3. Aggregate Variant-Level Features to Gene-Level

In [ ]:
print("\nAggregating variant-level features to gene-level...")

# Query to get gene-level aggregations
variant_agg_query = """
SELECT 
    gene_symbol,
    COUNT(*) as total_variants,
    SUM(CASE WHEN is_pathogenic = true THEN 1 ELSE 0 END) as pathogenic_count,
    AVG(CASE WHEN phylop_score IS NOT NULL THEN phylop_score ELSE 0 END) as avg_conservation,
    AVG(CASE WHEN cadd_phred IS NOT NULL THEN cadd_phred ELSE 0 END) as avg_cadd,
    SUM(CASE WHEN is_high_impact = 1 THEN 1 ELSE 0 END) as high_impact_count,
    SUM(CASE WHEN is_missense_variant = 1 THEN 1 ELSE 0 END) as missense_count,
    SUM(CASE WHEN is_loss_of_function = 1 THEN 1 ELSE 0 END) as lof_count
FROM clinical_ml_features
WHERE gene_symbol IS NOT NULL
GROUP BY gene_symbol
"""

variant_features = pd.read_sql(variant_agg_query, conn)
print(f"Aggregated features for {len(variant_features):,} genes")
print(f"\nSample:")
print(variant_features.head())

In [ ]:
# Calculate ratios
print("\nCalculating gene-level ratios...")

variant_features['pathogenic_ratio'] = (
    variant_features['pathogenic_count'] / variant_features['total_variants']
).fillna(0)

variant_features['high_impact_ratio'] = (
    variant_features['high_impact_count'] / variant_features['total_variants']
).fillna(0)

variant_features['lof_ratio'] = (
    variant_features['lof_count'] / variant_features['total_variants']
).fillna(0)

print("OK Ratios calculated")

---
## 4. Merge Gene-Level Features

In [ ]:
print("\nMerging pharmacogene and variant features...")

# Merge on gene_symbol
gene_features = pharmacogene_df.merge(
    variant_features,
    on='gene_symbol',
    how='left'
)

# Fill missing values
gene_features = gene_features.fillna(0)

print(f"\nMerged dataset: {gene_features.shape}")
print(f"Columns: {list(gene_features.columns)}")
print(f"\nSample:")
print(gene_features.head())

---
## 5. Filter to Genes with Druggability Scores

In [ ]:
print("\nFiltering to genes with druggability scores...")

# Keep only genes with known druggability scores (not 0)
druggable_genes = gene_features[
    (gene_features['druggability_score'] > 0) | 
    (gene_features['enhanced_druggability_score'] > 0)
].copy()

print(f"Genes with druggability scores: {len(druggable_genes):,}")
print(f"Total genes: {len(gene_features):,}")
print(f"Percentage: {len(druggable_genes)/len(gene_features)*100:.1f}%")

# Use enhanced_druggability_score as target (more genes have this)
druggable_genes['target'] = druggable_genes['enhanced_druggability_score'].copy()

print(f"\nTarget distribution:")
print(druggable_genes['target'].describe())

---
## 6. Exploratory Analysis of Target

In [ ]:
print("\nAnalyzing target variable (druggability score)...")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(druggable_genes['target'], bins=50, edgecolor='black')
axes[0].set_xlabel('Druggability Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Druggability Scores')
axes[0].grid(True, alpha=0.3)

# Boxplot
axes[1].boxplot(druggable_genes['target'])
axes[1].set_ylabel('Druggability Score')
axes[1].set_title('Druggability Score Boxplot')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'phase9' / '01_druggability_distribution.png', dpi=150)
plt.show()

print("OK Target analysis plot saved")

---
## 7. Feature Selection for Druggability Model

In [ ]:
print("\nSelecting features for druggability prediction...")

# Feature columns (exclude identifiers and target)
feature_cols = [
    'is_pharmacogene', 'is_drug_target', 'is_kinase', 'is_receptor', 'is_transporter',
    'variant_count', 'total_variants', 'pathogenic_count', 'pathogenic_ratio',
    'avg_conservation', 'avg_cadd', 'high_impact_count', 'high_impact_ratio',
    'missense_count', 'lof_count', 'lof_ratio'
]

# Check which features exist
available_features = [f for f in feature_cols if f in druggable_genes.columns]
missing_features = [f for f in feature_cols if f not in druggable_genes.columns]

print(f"Available features: {len(available_features)}")
if missing_features:
    print(f"Missing features: {missing_features}")

# Create feature matrix
X = druggable_genes[available_features].copy()
y = druggable_genes['target'].copy()

print(f"\nFeature matrix: {X.shape}")
print(f"Target: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")

---
## 8. Train/Val/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

print("\nSplitting data: 70% train, 15% val, 15% test...")

# First split: train + val (85%) vs test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=MODEL_CONFIG['test_size'],
    random_state=MODEL_CONFIG['random_state']
)

# Second split: train (70%) vs val (15%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=MODEL_CONFIG['val_size'] / (1 - MODEL_CONFIG['test_size']),
    random_state=MODEL_CONFIG['random_state']
)

print(f"\nTrain set: {X_train.shape}")
print(f"Val set:   {X_val.shape}")
print(f"Test set:  {X_test.shape}")

print(f"\nTarget statistics:")
print(f"  Train mean: {y_train.mean():.3f}")
print(f"  Val mean:   {y_val.mean():.3f}")
print(f"  Test mean:  {y_test.mean():.3f}")

---
## 9. Save Prepared Datasets

In [ ]:
print("\nSaving prepared datasets...")

# Create output directory
output_dir = DATA_DIR / 'ml' / 'phase9'
output_dir.mkdir(parents=True, exist_ok=True)

# Save datasets
datasets = {
    'train': {'X': X_train, 'y': y_train},
    'val': {'X': X_val, 'y': y_val},
    'test': {'X': X_test, 'y': y_test}
}

for name, data in datasets.items():
    with open(output_dir / f'druggability_{name}.pkl', 'wb') as f:
        pickle.dump(data, f)
    print(f"  Saved {name} set: {data['X'].shape}")

# Save feature names
with open(output_dir / 'druggability_features.txt', 'w') as f:
    f.write('\n'.join(available_features))

print(f"\nOK All datasets saved to {output_dir}")

---
## 10. Summary

In [ ]:
print("\n" + "="*80)
print("DRUGGABILITY FEATURE ENGINEERING SUMMARY")
print("="*80)

print(f"\nTotal genes processed: {len(gene_features):,}")
print(f"Genes with druggability scores: {len(druggable_genes):,}")
print(f"Features created: {len(available_features)}")

print(f"\nDataset sizes:")
print(f"  Train: {len(X_train):,} samples")
print(f"  Val:   {len(X_val):,} samples")
print(f"  Test:  {len(X_test):,} samples")

print(f"\nTarget (druggability score):")
print(f"  Range: [{y.min():.3f}, {y.max():.3f}]")
print(f"  Mean: {y.mean():.3f}")
print(f"  Std: {y.std():.3f}")

print("\n" + "="*80)
print("FILES CREATED")
print("="*80)
print(f"\nDatasets:")
print(f"  - druggability_train.pkl")
print(f"  - druggability_val.pkl")
print(f"  - druggability_test.pkl")
print(f"  - druggability_features.txt")

print(f"\nFigures:")
print(f"  - 01_druggability_distribution.png")

print("\n" + "="*80)
print("PHASE 9.1 COMPLETE - READY FOR MODEL TRAINING")
print("="*80)
print("\nNext: 09b_druggability_model_training.ipynb")

# Close connection
conn.close()